# Method 작성과 사용 예제

간단한 그룹별 조회 Method를 작성하고 실행합니다. 이어서 코드를 수정하고, 여러 조회를 조합한 분석과 DoWhy 분석을 만들어봅니다.

연습에는 준비된 예제 데이터를 사용합니다. 마지막에는 같은 Method를 실제 Cube 데이터에 적용합니다.

## 1. 준비

In [116]:
from pathlib import Path
import sys

repository = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/decision_layer").is_dir()
)
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))

import pandas as pd
from IPython.display import display

from decision_layer.dev import MethodSession
from decision_layer.methods import (
    Dimension, Entity, Estimate, Filter, Measure, Refused, Scope, Table, method,
)
import examples.methods.patterns.teaching_data as example_data


def show_table(data):
    if isinstance(data, dict):
        data = [{"field": key, "value": value} for key, value in data.items()]
    display(pd.DataFrame(data))

import importlib
import examples.methods.patterns.scenarios as practice_scenarios

importlib.reload(practice_scenarios)
importlib.reload(example_data)

<module 'examples.methods.patterns.teaching_data' from '/Users/hc.cho/Workspace/decision-layer/examples/methods/patterns/teaching_data.py'>

## 2. 그룹별 조회 Method 만들기

지표 여러 개를 하나 이상의 기준으로 나눠 조회하는 Method를 만듭니다. 첫 번째 지표 값이 큰 순서로 상위 `limit`개를 반환합니다.

In [117]:
@method("workshop.by_group", version="1.0.0", interpretation="descriptive")
async def my_by_group(ctx, *, selected_measures: list[Measure],
                      group_by: list[Dimension], limit: int = 3) -> Table:
    data = await ctx.aggregate(
        measures={f"value_{i}": ref for i, ref in enumerate(selected_measures, 1)},
        dimensions={f"group_{i}": ref for i, ref in enumerate(group_by, 1)},
        order=[(selected_measures[0], "desc")],
        limit=limit,
    )
    return Table.from_dataset(data, require_non_empty=True)

`selected_measures`와 `group_by`는 목록입니다. 지표나 분류 기준이 하나여도 `[경로]`처럼 넣습니다. `ctx.aggregate(...)`는 분류 기준의 조합마다 모든 지표를 조회합니다.

- `measures`: 결과 열을 `value_1`, `value_2` 순서로 만듭니다.
- `dimensions`: 분류 기준 열을 `group_1`, `group_2` 순서로 만듭니다.
- `order`, `limit`: 첫 번째 지표 값으로 정렬해 상위 `limit`개를 가져옵니다.

### AI 요청에서 Method 실행까지

예: “2001년 2월 store 364의 판매액을 department별 상위 3개 보여줘.”

AI에게 요청하면 semantic layer에서 필요한 지표와 분류 기준을 찾아 입력을 채웁니다. 이번 예제에서는 아래 실행 코드에 값을 직접 넣어봅니다.

- `selected_measures`: 판매액 지표 `cube://journey/transaction/receipts`를 목록에 넣습니다.
- `group_by`: department 분류 기준 `cube://journey/product/department`를 목록에 넣습니다.
- `limit`: 3
- `scope`: 2001년 2월, 거래 날짜 기준, store 364 조건

`cube://journey/transaction/receipts`는 `journey` 연결의 `transaction` 모델에 있는 `receipts` 지표를 뜻합니다. 

연습 코드에서는 준비된 예제 데이터의 `cube://practice/observations/value`를 사용합니다.

### 노트북에서 Method 실행해 보기

방금 만든 `my_by_group`을 준비된 예제 응답으로 시험합니다.

- `example_session("by_group")`: 예제 응답을 사용하는 연습용 session을 만듭니다.
- `register(my_by_group)`: 이 session에 Method를 등록합니다.
- `run(...)`: 지표, 그룹, 기간을 넣어 실행합니다.

실제 서비스에서 AI가 MCP로 요청하면, 서버에 미리 등록된 Method에 입력을 전달해 실행합니다.

In [118]:
group_session = await example_data.example_session("by_group")
group_session.register(my_by_group)

trial = await group_session.run(
    my_by_group,
    selected_measures=["cube://practice/observations/value"],
    group_by=["cube://practice/observations/group"],
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    limit=3,
)

assert trial.result.status == "success"
assert [row["value_1"] for row in trial.result.primary.data] == [30, 20, 10]
assert len(trial.attempts) == 1
show_table(trial.result.primary.data)

group_session.close()

,value_1,group_1
0,30,A
1,20,B
2,10,C


`group_session.run(...)`은 `scope`의 기간과 필터를 담은 `ctx`를 만들어 `my_by_group`을 호출합니다. `my_by_group`은 `ctx.aggregate(...)`로 그룹별 조회를 요청합니다. 연습용 session은 요청을 확인한 뒤 준비된 값 `30, 20, 10`을 반환합니다.


### 조회 기록 확인

요청한 기간과 필터와 실행 상태를 확인합니다.

In [119]:
show_table([
    {
        "query": index,
        "label": attempt.label,
        "status": attempt.status,
        "period": attempt.spec.time.date_range,
        "filters": [item.model_dump() for item in attempt.spec.filters],
    }
    for index, attempt in enumerate(trial.attempts, start=1)
])

,query,label,status,period,filters
0,1,None,success,"(2026-02-01, 2026-02-28)",[]


### 지표와 분류 기준 여러 개 넣기

같은 Method에 지표 두 개와 분류 기준 두 개를 넣습니다. `group_1`과 `group_2`의 조합마다 두 지표를 조회하고, 첫 번째 지표로 정렬합니다.

In [120]:
multi_session = await example_data.example_session("multi_group")
multi_session.register(my_by_group)

multi_trial = await multi_session.run(
    my_by_group,
    selected_measures=[
        "cube://practice/observations/value",
        "cube://practice/observations/before_value",
    ],
    group_by=[
        "cube://practice/observations/group",
        "cube://practice/observations/subject",
    ],
    scope=Scope(("2026-02-01", "2026-02-28"), "cube://practice/observations/date"),
    limit=3,
)

assert multi_trial.result.status == "success"
assert multi_trial.result.primary.data[0] == {
    "value_1": 30, "value_2": 25, "group_1": "A", "group_2": "X",
}
show_table(multi_trial.result.primary.data)
multi_session.close()

,value_1,value_2,group_1,group_2
0,30,25,A,X
1,20,15,B,Y
2,10,5,C,Z


## 3. 코드 수정: 순위 추가

조회 결과에 `rank`를 추가합니다. 

수정한 함수는 같은 Method 이름과 새 버전 `1.1.0`을 사용합니다.

In [121]:
@method("workshop.by_group", version="1.1.0", interpretation="descriptive")
async def my_ranked_by_group(ctx, *, selected_measures: list[Measure],
                             group_by: list[Dimension], limit: int = 3) -> Table:
    data = await ctx.aggregate(
        measures={f"value_{i}": ref for i, ref in enumerate(selected_measures, 1)},
        dimensions={f"group_{i}": ref for i, ref in enumerate(group_by, 1)},
        order=[(selected_measures[0], "desc")],
        limit=limit,
    )
    rows = data.records()
    for rank, row in enumerate(rows, start=1):
        row["rank"] = rank
    return Table(rows)

### 수정한 함수 실행

새 연습용 session에 수정한 Method만 등록해 실행합니다. 결과에 순위 `1, 2, 3`이 추가됩니다.

In [122]:
ranked_session = await example_data.example_session("by_group")
ranked_session.register(my_ranked_by_group)

edited_trial = await ranked_session.run(
    my_ranked_by_group,
    selected_measures=["cube://practice/observations/value"],
    group_by=["cube://practice/observations/group"],
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    limit=3,
)

assert edited_trial.result.status == "success"
assert [row["rank"] for row in edited_trial.result.primary.data] == [1, 2, 3]
assert [row["value_1"] for row in edited_trial.result.primary.data] == [30, 20, 10]
show_table(edited_trial.result.primary.data)

ranked_session.close()

,value_1,group_1,rank
0,30,A,1
1,20,B,2
2,10,C,3


## 4. 여러 조회 조합: Peer Comparison

한 함수에서 세 번 조회하고, 받은 값을 하나의 표로 합칩니다.

| 조회 | 대상 |
| --- | --- |
| subject | 그룹 X의 대상 A |
| peers | 그룹 X에서 A를 제외한 집단 |
| overall | 전체에서 A를 제외한 집단 |

`ctx.scope.filters`는 모든 조회에 적용되는 공통 조건입니다. 아래 함수는 비교 집단까지 대상 A로 제한되지 않도록 이 조건을 확인합니다.

In [123]:
@method(
    "workshop.peer",
    version="1.0.0",
    interpretation="descriptive",
)
async def my_peer(
    ctx,
    *,
    measure: Measure,
    subject: Dimension,
    category: Dimension,
    subject_value: str,
    category_value: str,
) -> Table:
    if any(item.member == subject for item in ctx.scope.filters):
        raise Refused("Keep the subject out of shared filters")

    same_category = Filter(
        member=category,
        operator="equals",
        values=[category_value],
    )
    selected = Filter(
        member=subject, 
        operator="equals", 
        values=[subject_value]
    )
    excluded = Filter(member=subject, operator="notEquals", values=[subject_value])

    subject_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[same_category, selected],
        label="subject",
    )
    peer_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[same_category, excluded],
        label="peers",
    )
    overall_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[excluded],
        label="overall",
    )

    subject_result = subject_data.scalar("value")
    peer_result = peer_data.scalar("value")
    overall_result = overall_data.scalar("value")

    return Table(
        [
            {
                "comparison": "peers",
                "subject": subject_result,
                "benchmark": peer_result,
                "difference": subject_result - peer_result,
            },
            {
                "comparison": "overall",
                "subject": subject_result,
                "benchmark": overall_result,
                "difference": subject_result - overall_result,
            },
        ],
        warnings=["Descriptive comparison; no causal claim or significance test."],
    )

### 등록하고 실행

대상 A와 그룹 X를 지정합니다. 준비된 응답은 `12, 8, 6`이므로 차이는 `4, 6`입니다.

In [124]:
peer_session = await example_data.example_session("peer")
peer_session.register(my_peer)

peer_trial = await peer_session.run(
    my_peer,
    measure="cube://practice/observations/rate",
    subject="cube://practice/observations/subject",
    category="cube://practice/observations/group",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    subject_value="A",
    category_value="X",
)

assert peer_trial.result.status == "success"
assert [row["difference"] for row in peer_trial.result.primary.data] == [4, 6]
assert len(peer_trial.attempts) == 3
show_table(peer_trial.result.primary.data)

peer_session.close()

,comparison,subject,benchmark,difference
0,peers,12,8,4
1,overall,12,6,6


### 세 조회 확인

각 조회의 label과 필터를 비교합니다. 직접 수정해보려면 결과에 `ratio`를 추가해보세요.

In [125]:
show_table([
    {
        "query": index,
        "label": attempt.label,
        "status": attempt.status,
        "filters": [item.model_dump() for item in attempt.spec.filters],
    }
    for index, attempt in enumerate(peer_trial.attempts, start=1)
])

,query,label,status,filters
0,1,subject,success,[{'member': 'cube://practice/observations/grou...
1,2,peers,success,[{'member': 'cube://practice/observations/grou...
2,3,overall,success,[{'member': 'cube://practice/observations/subj...


## 5. DoWhy 회귀 조정

DoWhy의 `backdoor.linear_regression`으로 이진 처치의 `0→1` 차이를 조정합니다.

| 입력 | 의미 |
| --- | --- |
| outcome | 결과 지표 |
| treatment | 0 또는 1인 처치 |
| confounders | 처치 전 조정 변수 |
| unit | 분석 단위 |

데이터 조건을 검사해도 식별 가정이 검증되는 것은 아닙니다. 이 예제는 연관성 결과를 보고하며 신뢰구간을 제공하지 않습니다.

### 계산 함수 작성

데이터 검사 → 전처리 → 지원 범위 확인 → DoWhy 추정 순서입니다. 같은 코드를 [binary_regression.py](binary_regression.py)에서도 볼 수 있습니다.

In [126]:
def estimate_binary_regression(frame, confounder_columns):
    """Estimate the 0-to-1 contrast under an explicitly assumed backdoor graph."""
    import networkx as nx
    import numpy as np
    import pandas as pd
    from dowhy import CausalModel
    from sklearn.linear_model import LogisticRegression

    # 1. Check the observation units, missing values and treatment groups.
    if (
        frame.empty
        or frame["unit"].isna().any()
        or frame["unit"].duplicated().any()
    ):
        raise Refused("Unique non-null analysis units are required")

    required_columns = ["outcome", "treatment", *confounder_columns]
    if not confounder_columns or frame[required_columns].isna().any().any():
        raise Refused(
            "Observed adjustment variables and a missing-data policy are required"
        )

    if (
        set(frame["treatment"].unique()) != {0, 1}
        or frame.groupby("treatment").size().min() < 30
    ):
        raise Refused(
            "This example requires binary treatment and at least 30 units per arm"
        )

    # 2. Bound encoding before allocating the matrix.
    encoded_width = sum(
        1 if pd.api.types.is_numeric_dtype(frame[column]) else max(frame[column].nunique() - 1, 0)
        for column in confounder_columns
    )
    if encoded_width == 0 or encoded_width > 100 or encoded_width * len(frame) > 2_000_000:
        raise Refused("Adjustment encoding is empty or exceeds the matrix bound")
    encoded = pd.get_dummies(
        frame[confounder_columns],
        drop_first=True,
        dtype=float,
    )
    if (
        not len(encoded.columns)
        or len(encoded.columns) > 100
        or encoded.size > 2_000_000
    ):
        raise Refused("Adjustment encoding is empty or exceeds the matrix bound")

    encoded.columns = [f"adjustment_{i}" for i in range(len(encoded.columns))]

    if (
        not np.isfinite(frame["outcome"].to_numpy(dtype=float)).all()
        or not np.isfinite(encoded.to_numpy()).all()
    ):
        raise Refused("Finite outcomes and adjustment variables are required")

    observations = pd.concat(
        [
            frame[["outcome", "treatment"]].reset_index(drop=True),
            encoded.reset_index(drop=True),
        ],
        axis=1,
    )

    predictor_columns = ["treatment", *encoded.columns]
    predictors = observations[predictor_columns].to_numpy(dtype=float)
    matrix = np.column_stack([np.ones(len(frame)), predictors])

    if (
        len(frame) <= matrix.shape[1]
        or np.linalg.matrix_rank(matrix) < matrix.shape[1]
    ):
        raise Refused(
            "The regression needs full-rank predictors and residual degrees of freedom"
        )

    # 3. Diagnose observed treatment support. This does not prove positivity.
    propensity = LogisticRegression(
        max_iter=1000,
        random_state=7,
    ).fit(encoded, frame["treatment"])
    probability = propensity.predict_proba(encoded)[:, 1]

    if np.any(probability < 0.05) or np.any(probability > 0.95):
        raise Refused(
            "Observed support fails this example's diagnostic; not a positivity proof"
        )

    # 4. Declare the assumed DAG and identify its backdoor estimand.
    edges = [("treatment", "outcome")]
    edges += [
        (column, target)
        for column in encoded.columns
        for target in ("treatment", "outcome")
    ]
    model = CausalModel(
        data=observations,
        treatment="treatment",
        outcome="outcome",
        graph=nx.DiGraph(edges),
    )
    estimand = model.identify_effect(proceed_when_unidentifiable=False)

    if not (estimand.estimands.get("backdoor") or {}).get("estimand"):
        raise Refused("No backdoor estimand is identified under the assumed graph")

    # 5. Fit the explicit binary regression contrast and a placebo diagnostic.
    estimate = model.estimate_effect(
        estimand,
        method_name="backdoor.linear_regression",
        control_value=0,
        treatment_value=1,
        target_units="ate",
        confidence_intervals=False,
    )
    placebo = model.refute_estimate(
        estimand,
        estimate,
        method_name="placebo_treatment_refuter",
        placebo_type="permute",
        num_simulations=20,
        random_state=7,
    )

    # 6. Return the estimate with its analysis settings and interpretation limits.
    return {
        "value": float(estimate.value),
        "estimator": "backdoor.linear_regression",
        "contrast": [0, 1],
        "target": "ate",
        "n_units": len(frame),
        "graph_edges": edges,
        "estimand": str(estimand),
        "placebo_difference": float(placebo.new_effect),
        "seed": 7,
        "support": {
            "min": float(probability.min()),
            "max": float(probability.max()),
        },
        "causal_claim": "not_established",
        "uncertainty": "not_reported",
    }

### Method로 연결

`ctx.entities(...)`로 `unit`별 데이터를 가져온 뒤, `ctx.compute(...)`로 위 계산 함수를 실행합니다.

In [127]:
@method(
    "workshop.binary_regression",
    version="1.0.0",
    interpretation="associational",
    execution="dataframe",
    requires_packages={
        "dowhy": "==0.13",
        "numpy": ">=1.26,<2",
        "pandas": ">=2.2,<3",
        "scikit-learn": ">=1.2,<1.7",
    },
)
async def my_binary_regression(
    ctx,
    *,
    outcome: Measure,
    treatment: Dimension,
    confounders: list[Dimension | Measure],
    unit: Entity,
) -> Estimate:
    # Extract governed entity data, then run ordinary Python calculation code.
    observations = await ctx.entities(
        entity=unit,
        measures={"outcome": outcome},
        dimensions={
            "treatment": treatment,
            "confounders": confounders,
        },
    )
    payload = await ctx.compute(
        estimate_binary_regression,
        observations.frame,
        observations.columns("confounders"),
    )

    return Estimate(
        payload["value"],
        details={
            **payload,
            "assumptions_status": "not_verified",
        },
        warnings=[
            "Unverified assumptions and placebo diagnostics do not establish "
            "causality; no interval is reported."
        ],
    )

### 등록하고 실행

결과, 처치, 조정 변수, 분석 단위를 연결합니다. 예제 데이터의 생성식은 `Y = 2*T + 3*W + noise`이며 기대값은 약 `2`입니다.

In [128]:
regression_session = await example_data.example_session("dowhy")
regression_session.register(my_binary_regression)

dowhy_trial = await regression_session.run(
    my_binary_regression,
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/treated",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
)

assert dowhy_trial.result.status == "success"
assert abs(dowhy_trial.result.primary.data["value"] - 2) < 0.15
show_table(dowhy_trial.result.primary.data)

regression_session.close()


/Users/hc.cho/Workspace/decision-layer/.venv/lib/python3.11/site-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
/Users/hc.cho/Workspace/decision-layer/.venv/lib/python3.11/site-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
/Users/hc.cho/Workspace/decision-layer/.venv/lib/python3.11/site-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is d

,field,value
0,value,2.023501
1,estimator,backdoor.linear_regression
2,contrast,"[0, 1]"
3,target,ate
4,n_units,2000
5,graph_edges,"[[treatment, outcome], [adjustment_0, treatmen..."
6,estimand,Estimand type: EstimandType.NONPARAMETRIC_ATE\...
7,placebo_difference,0.002933
8,seed,7
9,support,"{'min': 0.25189573942649124, 'max': 0.74455237..."


### 일반 Python에서도 사용

같은 계산 함수에 DataFrame을 직접 전달합니다. Method 실행과 같은 값이 나오는지 확인합니다.

In [129]:
frame = example_data.binary_regression_frame()
standalone_result = estimate_binary_regression(frame, ["baseline"])

assert abs(standalone_result["value"] - 2) < 0.15
assert standalone_result["value"] == dowhy_trial.result.primary.data["value"]
show_table(standalone_result)


/Users/hc.cho/Workspace/decision-layer/.venv/lib/python3.11/site-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
/Users/hc.cho/Workspace/decision-layer/.venv/lib/python3.11/site-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
/Users/hc.cho/Workspace/decision-layer/.venv/lib/python3.11/site-packages/dowhy/causal_estimators/regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is d

,field,value
0,value,2.023501
1,estimator,backdoor.linear_regression
2,contrast,"[0, 1]"
3,target,ate
4,n_units,2000
5,graph_edges,"[(treatment, outcome), (adjustment_0, treatmen..."
6,estimand,Estimand type: EstimandType.NONPARAMETRIC_ATE\...
7,placebo_difference,0.002933
8,seed,7
9,support,"{'min': 0.25189573942649124, 'max': 0.74455237..."


### 실패 확인

분석 단위가 중복된 연습 데이터를 넣습니다. Method는 추정을 거절하고 이유를 반환합니다.

In [130]:
refusal_session = await example_data.duplicate_unit_session()
refusal_session.register(my_binary_regression)

refused_trial = await refusal_session.run(
    my_binary_regression,
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/treated",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
)

assert refused_trial.result.status == "refused"
show_table([
    {"status": refused_trial.result.status, "reason": warning}
    for warning in refused_trial.result.warnings
])
refusal_session.close()


,status,reason
0,refused,Entity observations need unique non-null analy...


## 6. 실제 Cube에서 사용

연결 셀의 `RUN_LIVE`를 `True`로 바꾸세요. 기본 URL과 secret은 로컬 샘플 설정입니다. 실제 catalog에서 모델과 항목을 가리키는 경로를 확인한 뒤 위 함수를 그대로 실행합니다.

In [131]:
import os
from decision_layer.semantic.credentials import ServiceCredentials

RUN_LIVE = False
live = None

if RUN_LIVE:
    live = await MethodSession.cube(
        os.environ.get("DL_JOURNEY_CUBE_URL", "http://127.0.0.1:4000/cubejs-api/v1"),
        ServiceCredentials(
            os.environ.get(
                "DL_JOURNEY_CUBE_SECRET",
                "local-example-secret-change-me-0123456789",
            ),
            (),
        ),
        instance="journey",
    )

show_table([{
    "status": "connected" if live is not None else "skipped",
    "reason": "Cube connected" if live is not None else "Set RUN_LIVE = True in this connection cell",
}])

,status,reason
0,skipped,Set RUN_LIVE = True in this connection cell


### 객체 확인

연결한 Cube에서 receipts 지표와 department를 찾습니다. catalog는 사용 가능한 객체 목록이며, Method 안에서도 `ctx.catalog`로 확인할 수 있습니다.

In [132]:
if live is not None:
    objects = [
        *live.catalog.search(kind="measure", text="receipts"),
        *live.catalog.search(kind="dimension", text="department"),
    ]
    show_table([
        {"title": obj.title, "kind": obj.kind, "path": obj.ref}
        for obj in objects
    ])
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

,status,reason
0,skipped,Enable the Cube connection above


### 그룹별 조회

판매액을 department별로 조회합니다. `limit`이나 기간을 바꿔보세요.

In [133]:
if live is not None:
    live.register(my_by_group, replace=True)
    live_group_trial = await live.run(
        my_by_group,
        selected_measures=["cube://journey/transaction/receipts"],
        group_by=["cube://journey/product/department"],
        scope=Scope(
            ("2001-01-01", "2001-06-30"),
            "cube://journey/transaction/analysis_date",
        ),
        limit=3,
    )
    show_table(live_group_trial.result.primary.data)
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

,status,reason
0,skipped,Enable the Cube connection above


### 집단 비교

GROCERY의 store 364를 동종 집단과 전체 집단과 비교합니다. 실제 존재하는 다른 store나 department로 바꿔보세요.

In [134]:
if live is not None:
    live.register(my_peer, replace=True)
    live_peer_trial = await live.run(
        my_peer,
        measure="cube://journey/transaction/coupon_line_rate",
        subject="cube://journey/transaction/store",
        category="cube://journey/product/department",
        scope=Scope(
            ("2001-01-01", "2001-06-30"),
            "cube://journey/transaction/analysis_date",
        ),
        subject_value="364",
        category_value="GROCERY",
    )
    show_table(live_peer_trial.result.primary.data)
    show_table([
        {"query": index, "label": attempt.label, "status": attempt.status}
        for index, attempt in enumerate(live_peer_trial.attempts, start=1)
    ])
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

,status,reason
0,skipped,Enable the Cube connection above


### 연결 정리

로컬 session을 닫습니다. 서버에 제공할 Method는 검토한 Python 파일에 저장하고 서버에서 등록합니다.

In [135]:
if live is not None:
    live.close()
    live = None

show_table([{"status": "closed"}])

,status
0,closed
